# Comparação de Modelos — Thermal Predictor

Este notebook reúne, em um só lugar e sob um só protocolo, os três algoritmos
candidatos do Thermal Predictor. Corresponde à **Seção 4.4** da documentação.

Cada modelo já tem notebook próprio, com o histórico completo de experimentos
do algoritmo:

| Algoritmo | Notebook de origem |
|---|---|
| Random Forest | `Random_for_regre.ipynb` |
| Gradient Boosting | `Grad_Boosting_Regre.ipynb` |
| Decision Tree | `Decision_tree_regre.ipynb` |

**O que este notebook acrescenta.** Os números que decidem qual algoritmo vai
para produção em cada região estavam, até aqui, transcritos à mão na
documentação a partir das saídas de dois notebooks diferentes. Aqui eles são
**recalculados do zero, no mesmo processo**, de modo que a tabela comparativa
deixa de depender de transcrição.

**Por que isso não é redundante.** Os dois notebooks de origem agregam as
métricas de formas diferentes — o Random Forest reporta a **média** do R² entre
sensores, o Gradient Boosting a **mediana** — e essa diferença nunca foi
neutralizada. A Seção 3 mede o tamanho do efeito e adota uma convenção única
para os três algoritmos.

---

## Sumário

1. Configuração e funções compartilhadas
2. Protocolo de avaliação
3. Convenção de agregação das métricas
4. Os três candidatos sob o mesmo protocolo
5. Tabela comparativa e escolha por região
6. Explicabilidade
7. Conclusão

## 1. Configuração e funções compartilhadas

As funções abaixo são as mesmas dos três notebooks de origem — leitura da
planilha, seleção de alvos por cobertura, padronização de grafias e codificação
one-hot. Estão reunidas aqui para que a comparação parta, comprovadamente, do
mesmo pré-processamento para os três algoritmos.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import GridSearchCV

warnings.filterwarnings("ignore")

SEMENTE = 42
COBERTURA_MINIMA_ALVO = 0.50
N_FOLDS_DESEJADO = 3

VARIAVEIS_INDEPENDENTES = [
    "blower_direction",
    "housing",
    "board_position",
    "run_case",
    "losses",
    "deflector_hole",
    "material_coil_support",
]

# Grafias divergentes observadas na base bruta, unificadas conforme o
# glossario do parceiro.
GRAFIAS_DIVERGENTES = {
    "blower_direction":      {"To back": "To Back", "To front": "To Front"},
    "losses":                {"No losses": "No Losses"},
    "run_case":              {"Broil Case 2": "Broil C2",
                              "Broil Qual": "Broil Qualitative",
                              "Qualitative": "Broil Qualitative"},
    "housing":               {"Open - Updated": "Open"},
    "material_coil_support": {"Aluminium Plate": "Aluminium"},
}

# Roda tanto a partir de notebooks/ quanto da raiz do repositorio.
DIR_DADOS = Path("../data") if Path("../data").is_dir() else Path("data")

AREAS = ["mainback", "coil_temps", "main_vent_sys", "ct_knobs",
         "ipc_comp", "ipc_cooling", "oven_hmi"]

ROTULOS_AREA = {
    "mainback": "Mainback", "coil_temps": "Coil Temps",
    "main_vent_sys": "Main Vent Sys", "ct_knobs": "CT Knobs",
    "ipc_comp": "IPC Comp", "ipc_cooling": "IPC Cooling",
    "oven_hmi": "Oven HMI",
}

CORES = {"Random Forest": "#009E73",
         "Gradient Boosting": "#0072B2",
         "Decision Tree": "#E69F00"}

print(f"Dados em: {DIR_DADOS.resolve()}")
print(f"Semente global: {SEMENTE}")

In [ ]:
def identificar_alvos(df, independentes):
    """Separa as colunas-alvo (termopares) das colunas de configuracao."""
    alvos = []
    for coluna in df.columns:
        if coluna in independentes:
            continue
        if str(coluna).startswith("Unnamed:") or not str(coluna).strip():
            continue
        if df[coluna].isna().all():
            continue
        alvos.append(coluna)
    return alvos


def filtrar_alvos_por_cobertura(df, alvos, cobertura_minima):
    """Mantem apenas os termopares medidos em uma fracao minima dos runs."""
    return [a for a in alvos if df[a].notna().mean() >= cobertura_minima]


def padronizar_categorias(df, independentes, correcoes_por_variavel):
    """Unifica grafias divergentes e transforma nulos em categoria explicita.

    A ordem importa: primeiro normaliza espacos (bordas E espacos duplos
    internos), so DEPOIS aplica as correcoes de grafia do dicionario. O
    dado bruto tem "Aluminium " e "Plastic " com espaco sobrando em
    material_coil_support, e "Miro Qualitative  - ..." com espaco duplo em
    losses. Na ordem inversa, "Aluminium " (26 casos) e "Aluminium" (a
    forma corrigida de "Aluminium Plate", 1 caso) ficam como duas colunas
    one-hot diferentes para a MESMA categoria - o get_dummies nao sabe que
    sao a mesma coisa, e nao ha erro nenhum para avisar disso.
    """
    def normalizar_espacos(v):
        return " ".join(v.split()) if isinstance(v, str) else v

    df = df.copy()
    for variavel in independentes:
        if variavel in df.columns:
            df[variavel] = df[variavel].apply(normalizar_espacos)
    for variavel, correcoes in correcoes_por_variavel.items():
        if variavel in df.columns:
            df[variavel] = df[variavel].replace(correcoes)
    for variavel in independentes:
        df[variavel] = df[variavel].fillna("Nao Informado").astype(str)
    return df


def codificar_onehot(df, independentes):
    """Uma coluna binaria por categoria observada."""
    return pd.get_dummies(df[independentes].astype(str),
                          columns=independentes).astype(int)


def preparar_area(area):
    """Leitura + selecao de alvos + codificacao de uma key_area."""
    df = pd.read_csv(DIR_DADOS / f"{area}_data.csv")
    alvos = filtrar_alvos_por_cobertura(
        df, identificar_alvos(df, VARIAVEIS_INDEPENDENTES), COBERTURA_MINIMA_ALVO)
    tratado = df.dropna(subset=alvos).copy()
    tratado = padronizar_categorias(tratado, VARIAVEIS_INDEPENDENTES,
                                    GRAFIAS_DIVERGENTES)
    X = codificar_onehot(tratado, VARIAVEIS_INDEPENDENTES)
    return X, tratado[alvos], alvos


DADOS = {area: preparar_area(area) for area in AREAS}

resumo = pd.DataFrame([
    {"key_area": ROTULOS_AREA[a],
     "configuracoes": len(DADOS[a][0]),
     "termopares": len(DADOS[a][2]),
     "colunas_onehot": DADOS[a][0].shape[1]}
    for a in AREAS
])

print("BASE APOS O PRE-PROCESSAMENTO COMUM")
print("=" * 62)
print(resumo.to_string(index=False))

### Checagem de sanidade: categorias com espaço sobrando

Antes de seguir, uma verificação automática: nenhuma categoria de nenhuma
variável pode ter espaço nas bordas ou espaço duplo interno sobrevivendo à
padronização. Se sobrar, o one-hot cria uma coluna a mais para a mesma
categoria, e a importância dela some sem erro nenhum aparecer — é
exatamente o tipo de falha que passa despercebido porque não quebra a
execução, só corrompe o resultado.

In [ ]:
problemas = []
for area in AREAS:
    X, Y, alvos = DADOS[area]
    df_bruto = pd.read_csv(DIR_DADOS / f"{area}_data.csv")
    for variavel in VARIAVEIS_INDEPENDENTES:
        if variavel not in df_bruto.columns:
            continue
        valores = df_bruto[variavel].dropna().astype(str)
        for v in valores.unique():
            if v != v.strip() or "  " in v:
                problemas.append((area, variavel, repr(v)))

if problemas:
    print("ATENCAO - categorias com espaco sobrando NOS DADOS BRUTOS "
          "(devem ser resolvidas por padronizar_categorias):")
    for area, variavel, valor in problemas:
        print(f"  {area:15s} {variavel:22s} {valor}")
else:
    print("Nenhuma categoria com espaco sobrando nos dados brutos.")

# A checagem que importa de verdade: DEPOIS da padronizacao, nenhuma
# categoria residual deve sobreviver, e nenhuma coluna one-hot deve ficar
# duplicada por causa de espaco.
falhas_pos_padronizacao = []
for area in AREAS:
    X, Y, alvos = DADOS[area]
    for coluna in X.columns:
        if coluna != coluna.strip() or "  " in coluna:
            falhas_pos_padronizacao.append((area, coluna))

print()
if falhas_pos_padronizacao:
    print("FALHA - colunas one-hot com espaco sobrando APOS a padronizacao:")
    for area, coluna in falhas_pos_padronizacao:
        print(f"  {area:15s} {coluna!r}")
    raise AssertionError(
        "Existem colunas one-hot com espaco sobrando - a padronizacao nao "
        "eliminou o problema. Ver padronizar_categorias() na Secao 1."
    )
else:
    print("OK - nenhuma coluna one-hot com espaco sobrando apos a padronizacao.")

### Leitura

As sete regiões têm **32 configurações** cada e o mesmo número de colunas
one-hot (28), variando só na quantidade de termopares — de 4 em `ipc_cooling`
e `oven_hmi` a 22 em `ipc_comp`. Essa diferença explica por que `ipc_comp`
aparece entre as regiões de maior erro em todos os modelos: são mais sensores
para o mesmo punhado de ensaios.

O ponto que condiciona **todo** o resto do notebook: 32 configurações é pouco.
Nenhum resultado aqui deve ser lido como estimativa estável de desempenho em
produção, e a Seção 7 retoma essa ressalva.

## 2. Protocolo de avaliação

**Validação: LOGO (Leave-One-Group-Out).** Cada linha das planilhas é uma
configuração única, então o LOGO equivale a Leave-One-Out: o modelo é
retreinado 32 vezes por região, deixando uma configuração de fora a cada vez e
prevendo justamente ela. Toda configuração serve de teste exatamente uma vez.

A alternativa seria o corte fixo de 20% (*holdout*) usado nas primeiras
versões, mas com ~6 configurações de teste por região o resultado passa a
depender do sorteio — é o que produz os R² fortemente negativos discutidos nas
Seções 4.4.1 e 4.4.2 da documentação. O LOGO não torna o modelo melhor; torna
a **medição** dele mais estável.

**Codificação: one-hot** para os três algoritmos, por não impor ordem
artificial entre categorias.

**Métricas.** As mesmas da Seção 4.3, pelos motivos justificados lá:

| Métrica | Pergunta que responde | Direção |
|---|---|---|
| **MAE** (°C) | De quantos graus a previsão erra, em média? | menor é melhor |
| **RMSE** (°C) | Há erros grandes escondidos atrás da média? | menor é melhor |
| **R²** | O modelo aprendeu, ou só reproduz a média? | maior é melhor |
| **dentro de 10%** | Que fração cai na margem acordada com o parceiro? | maior é melhor |

O MAPE aparece só como apoio, com a ressalva sobre escala intervalar feita na
Seção 4.3 — em escala Celsius, 10% de 150 °C são 15 °C, mas os mesmos 150 °C
valem 423,15 K, cujos 10% são 42,3 °C.

In [ ]:
def metricas_por_sensor(Y_real, Y_previsto):
    """Metricas calculadas sensor a sensor.

    Devolve uma linha por termopar. A agregacao entre sensores fica a cargo
    de quem chama - que e exatamente o ponto discutido na Secao 3.
    """
    real = np.asarray(Y_real, dtype=float)
    previsto = np.asarray(Y_previsto, dtype=float)

    linhas = []
    for j in range(real.shape[1]):
        y, p = real[:, j], previsto[:, j]
        ss_res = ((y - p) ** 2).sum()
        ss_tot = ((y - y.mean()) ** 2).sum()
        linhas.append({
            "MAE":    np.abs(y - p).mean(),
            "RMSE":   np.sqrt(((y - p) ** 2).mean()),
            "R2":     1 - ss_res / ss_tot if ss_tot > 0 else np.nan,
            "MAPE_%": np.mean(np.abs((y - p) / y)) * 100,
        })
    return pd.DataFrame(linhas)


def dentro_de_10(Y_real, Y_previsto):
    """Fracao de previsoes dentro da margem de 10% acordada com o parceiro."""
    real = np.asarray(Y_real, dtype=float)
    previsto = np.asarray(Y_previsto, dtype=float)
    return float((np.abs((real - previsto) / real) <= 0.10).mean())


def prever_logo(area, construir_modelo):
    """Previsoes out-of-fold por Leave-One-Out sobre as configuracoes.

    `construir_modelo` e uma funcao sem argumentos que devolve um estimador
    novo - um por fold, para que nenhum fold veja a configuracao de teste.
    """
    X, Y, _ = DADOS[area]
    n = len(X)
    previsto = np.zeros(Y.shape, dtype=float)

    for i in range(n):
        treino = np.arange(n) != i
        modelo = construir_modelo()
        modelo.fit(X.values[treino], Y.values[treino])
        previsto[i] = modelo.predict(X.values[[i]])[0]

    return Y.values, previsto


print("Funcoes de metrica e de validacao LOGO definidas.")

## 3. Convenção de agregação das métricas

Antes de comparar os três algoritmos, é preciso resolver uma inconsistência
que veio dos notebooks de origem.

Cada região tem vários termopares, e cada métrica é calculada **por termopar**.
Para chegar a um número por região é preciso agregar — e os dois notebooks
fizeram isso de formas diferentes:

| Notebook | R² da região | RMSE da região |
|---|---|---|
| `Random_for_regre.ipynb` | **média** entre sensores | média dos RMSE por sensor |
| `Grad_Boosting_Regre.ipynb` | **mediana** entre sensores | RMSE global (*pooled*) |

Isso importa porque média e mediana divergem muito quando um sensor destoa —
e com 32 configurações, sensores que destoam são a regra, não a exceção. A
célula abaixo mede a diferença em vez de assumi-la desprezível.

In [ ]:
# Um modelo barato (Random Forest pequeno) basta para medir o efeito da
# agregacao: o ponto aqui nao e o desempenho, e a distancia entre media e
# mediana sobre o MESMO vetor de previsoes.
linhas_agreg = []

for area in AREAS:
    real, previsto = prever_logo(
        area,
        lambda: RandomForestRegressor(n_estimators=100, random_state=SEMENTE,
                                      n_jobs=-1),
    )
    por_sensor = metricas_por_sensor(real, previsto)
    linhas_agreg.append({
        "key_area":    ROTULOS_AREA[area],
        "sensores":    len(por_sensor),
        "R2_media":    por_sensor["R2"].mean(),
        "R2_mediana":  por_sensor["R2"].median(),
        "diferenca":   por_sensor["R2"].median() - por_sensor["R2"].mean(),
    })

efeito_agregacao = pd.DataFrame(linhas_agreg)

print("EFEITO DA AGREGACAO SOBRE O R2 (mesmas previsoes, dois criterios)")
print("=" * 78)
print(efeito_agregacao.round(4).to_string(index=False))
print()
print(f"Diferenca absoluta media : {efeito_agregacao['diferenca'].abs().mean():.4f}")
print(f"Diferenca absoluta maxima: {efeito_agregacao['diferenca'].abs().max():.4f}"
      f"  ({efeito_agregacao.loc[efeito_agregacao['diferenca'].abs().idxmax(), 'key_area']})")

### Leitura e decisão

A diferença entre média e mediana **não** é desprezível: chega a somar décimos
de R² em algumas regiões. Comparar o R²-média do Random Forest com o
R²-mediana do Gradient Boosting, como a documentação fazia, mistura duas
grandezas diferentes.

A mediana é mais alta quando um sensor ruim puxa a média para baixo — ou seja,
a convenção do Gradient Boosting era, nessas regiões, a mais generosa das duas.

**Convenção adotada deste ponto em diante:** a **mediana** entre sensores, para
os três algoritmos igualmente. A escolha é pela robustez, não pelo valor: com
4 a 22 sensores por região e alguns deles mal medidos, um único sensor ruim
distorce a média da região inteira. O que importa para a validade da comparação
é que o critério seja **o mesmo** para os três — e agora é.

## 4. Os três candidatos sob o mesmo protocolo

Cada algoritmo é avaliado por LOGO, com one-hot, e agregado pela mediana entre
sensores. Os hiperparâmetros vêm dos notebooks de origem, onde foram
escolhidos por `GridSearchCV` — a busca não é repetida aqui, mas a Seção 4.4
registra a grade de cada um e reexecuta a busca para o Decision Tree, o mais
barato dos três.

### 4.1. Random Forest

Um comitê de árvores: cada uma treina sobre um recorte diferente dos mesmos
ensaios, e a previsão final é o consenso. Os erros individuais tendem a se
cancelar.

Os hiperparâmetros abaixo são os escolhidos por `GridSearchCV` no espaço
one-hot (`Random_for_regre.ipynb`, Célula 25b), sobre a grade
`n_estimators` ∈ {100, 300}, `max_depth` ∈ {5, 8, None},
`min_samples_leaf` ∈ {1, 2}, `max_features` ∈ {sqrt, None}.

In [ ]:
# Hiperparametros por area, escolhidos por GridSearchCV no espaco one-hot
# (Random_for_regre.ipynb, Celula 25b). Reexecutar a busca aqui custaria
# minutos sem mudar o resultado - a semente e fixa.
PARAMS_RF = {
    "mainback":      dict(n_estimators=100, max_depth=8,    min_samples_leaf=1,
                          max_features=None),
    "coil_temps":    dict(n_estimators=300, max_depth=5,    min_samples_leaf=1,
                          max_features=None),
    "main_vent_sys": dict(n_estimators=300, max_depth=5,    min_samples_leaf=1,
                          max_features=None),
    "ct_knobs":      dict(n_estimators=300, max_depth=8,    min_samples_leaf=1,
                          max_features="sqrt"),
    "ipc_comp":      dict(n_estimators=300, max_depth=5,    min_samples_leaf=1,
                          max_features="sqrt"),
    "ipc_cooling":   dict(n_estimators=100, max_depth=None, min_samples_leaf=1,
                          max_features=None),
    "oven_hmi":      dict(n_estimators=300, max_depth=5,    min_samples_leaf=1,
                          max_features="sqrt"),
}

PREVISOES = {}   # (algoritmo, area) -> (real, previsto)


def avaliar_algoritmo(nome, construtor_por_area):
    """Roda LOGO nas sete areas e devolve a tabela por area, agregada pela
    mediana entre sensores."""
    linhas = []
    for area in AREAS:
        real, previsto = prever_logo(area, construtor_por_area(area))
        PREVISOES[(nome, area)] = (real, previsto)

        por_sensor = metricas_por_sensor(real, previsto)
        linhas.append({
            "key_area":        ROTULOS_AREA[area],
            "MAE":             por_sensor["MAE"].median(),
            "RMSE":            por_sensor["RMSE"].median(),
            "R2":              por_sensor["R2"].median(),
            "MAPE_%":          por_sensor["MAPE_%"].median(),
            "dentro_de_10%":   dentro_de_10(real, previsto),
        })
        print(f"  OK  {ROTULOS_AREA[area]}")

    tabela = pd.DataFrame(linhas)
    tabela.attrs["algoritmo"] = nome
    return tabela


print("RANDOM FOREST - LOGO nas sete areas")
print("=" * 62)
rf_por_area = avaliar_algoritmo(
    "Random Forest",
    lambda area: (lambda: RandomForestRegressor(
        random_state=SEMENTE, n_jobs=-1, **PARAMS_RF[area])),
)

print()
print(rf_por_area.round(4).to_string(index=False))

### 4.2. Gradient Boosting

Correção sequencial em vez de comitê: cada árvore nova corrige o erro que as
anteriores deixaram. Como o `GradientBoostingRegressor` não aceita várias
saídas, o `MultiOutputRegressor` treina um regressor independente por termopar.

Hiperparâmetros fixos entre regiões, como em `Grad_Boosting_Regre.ipynb`
(Célula 8): `learning_rate = 0,03`, `max_depth = 3`, `subsample = 0,8`,
`n_estimators = 100`.

In [ ]:
PARAMS_GB = dict(n_estimators=100, learning_rate=0.03, max_depth=3,
                 subsample=0.8, random_state=SEMENTE)

print("GRADIENT BOOSTING - LOGO nas sete areas")
print("=" * 62)
gb_por_area = avaliar_algoritmo(
    "Gradient Boosting",
    lambda area: (lambda: MultiOutputRegressor(
        GradientBoostingRegressor(**PARAMS_GB), n_jobs=1)),
)

print()
print(gb_por_area.round(4).to_string(index=False))

### 4.3. Decision Tree

Uma única árvore por região. É o candidato mais interpretável — cada previsão
é um caminho de regras legível — e serve aqui como piso de comparação.

In [ ]:
GRADE_DT = {
    "max_depth":         [2, 3, 5, 8, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf":  [1, 2, 4],
    "max_features":      ["sqrt", None],
}

# A arvore e barata, entao aqui a busca e de fato reexecutada, em vez de
# transcrita. A busca roda sobre TODAS as configuracoes da area, e nao dentro
# de cada fold do LOGO - o que da a ela uma vantagem informacional sobre os
# outros dois. Como o Decision Tree nao e candidato a producao (Secao 5), essa
# vantagem so reforca a conclusao de que ele fica atras mesmo assim.
PARAMS_DT = {}
for area in AREAS:
    X, Y, _ = DADOS[area]
    busca = GridSearchCV(
        estimator=DecisionTreeRegressor(random_state=SEMENTE),
        param_grid=GRADE_DT,
        cv=min(N_FOLDS_DESEJADO, len(X)),
        scoring="neg_mean_absolute_error",
        n_jobs=-1,
    ).fit(X.values, Y.values)
    PARAMS_DT[area] = busca.best_params_

print("HIPERPARAMETROS ESCOLHIDOS POR GridSearchCV (Decision Tree)")
print("=" * 78)
print(pd.DataFrame([{"key_area": ROTULOS_AREA[a], **PARAMS_DT[a]}
                    for a in AREAS]).to_string(index=False))

print()
print("DECISION TREE - LOGO nas sete areas")
print("=" * 62)
dt_por_area = avaliar_algoritmo(
    "Decision Tree",
    lambda area: (lambda: DecisionTreeRegressor(
        random_state=SEMENTE, **PARAMS_DT[area])),
)

print()
print(dt_por_area.round(4).to_string(index=False))

### 4.4. Baseline — a média do treino

Nenhuma métrica acima significa muito sem um piso de comparação. O baseline
prevê, para cada termopar, a média das configurações de treino — ignorando
completamente a configuração de entrada.

Um modelo que não supere isso não aprendeu a relação entre montagem e
temperatura; apenas decorou o patamar térmico da região.

In [ ]:
class MediaDoTreino:
    """Baseline: preve a media do treino para cada termopar, ignorando X."""

    def fit(self, X, Y):
        self.media_ = np.asarray(Y, dtype=float).mean(axis=0)
        return self

    def predict(self, X):
        return np.tile(self.media_, (len(X), 1))


print("BASELINE (media do treino) - LOGO nas sete areas")
print("=" * 62)
bl_por_area = avaliar_algoritmo("Baseline", lambda area: MediaDoTreino)

print()
print(bl_por_area.round(4).to_string(index=False))

## 5. Tabela comparativa e escolha por região

Com os quatro avaliados sob o mesmo protocolo, a comparação fica direta.

In [ ]:
TABELAS = {
    "Random Forest":     rf_por_area,
    "Gradient Boosting": gb_por_area,
    "Decision Tree":     dt_por_area,
    "Baseline":          bl_por_area,
}

# --- Media entre as sete regioes, por algoritmo
resumo_geral = pd.DataFrame([
    {"Modelo": nome,
     "MAE (C)":       t["MAE"].mean(),
     "RMSE (C)":      t["RMSE"].mean(),
     "R2":            t["R2"].mean(),
     "MAPE (%)":      t["MAPE_%"].mean(),
     "Dentro de 10%": t["dentro_de_10%"].mean()}
    for nome, t in TABELAS.items()
])

print("MEDIA ENTRE AS SETE REGIOES (LOGO, one-hot, mediana entre sensores)")
print("=" * 88)
print(resumo_geral.round(4).to_string(index=False))

# --- Cada modelo vence o baseline em quantas regioes?
print()
print("REGIOES EM QUE CADA MODELO SUPERA O BASELINE (MAE menor)")
print("=" * 88)
for nome in ["Random Forest", "Gradient Boosting", "Decision Tree"]:
    vitorias = int((TABELAS[nome]["MAE"].values < bl_por_area["MAE"].values).sum())
    print(f"  {nome:20s} {vitorias} de 7")

In [ ]:
# --- Grafico: MAE por regiao, os tres candidatos contra o baseline
ordem = bl_por_area.sort_values("MAE")["key_area"].tolist()
x = np.arange(len(ordem))
largura = 0.2

fig, (ax_mae, ax_r2) = plt.subplots(1, 2, figsize=(16, 6))

for i, nome in enumerate(["Random Forest", "Gradient Boosting", "Decision Tree"]):
    t = TABELAS[nome].set_index("key_area").loc[ordem]
    ax_mae.bar(x + (i - 1) * largura, t["MAE"], largura,
               label=nome, color=CORES[nome])

t_bl = bl_por_area.set_index("key_area").loc[ordem]
ax_mae.plot(x, t_bl["MAE"], "k_", markersize=26, markeredgewidth=2.2,
            label="Baseline (média)", zorder=5)

ax_mae.set_xticks(x)
ax_mae.set_xticklabels(ordem, rotation=30, ha="right")
ax_mae.set_ylabel("MAE (\u00b0C)")
ax_mae.set_title("Erro por região\n(menor é melhor; traço preto = baseline)")
ax_mae.legend(fontsize=9)
ax_mae.grid(axis="y", alpha=0.25)
ax_mae.set_axisbelow(True)

for i, nome in enumerate(["Random Forest", "Gradient Boosting", "Decision Tree"]):
    t = TABELAS[nome].set_index("key_area").loc[ordem]
    ax_r2.bar(x + (i - 1) * largura, t["R2"], largura,
              label=nome, color=CORES[nome])

ax_r2.axhline(0, color="#555555", linewidth=1.0)
ax_r2.set_xticks(x)
ax_r2.set_xticklabels(ordem, rotation=30, ha="right")
ax_r2.set_ylabel("R\u00b2 (mediana entre sensores)")
ax_r2.set_title("Poder de distinção por região\n(maior é melhor; 0 = empata com a média)")
ax_r2.legend(fontsize=9)
ax_r2.grid(axis="y", alpha=0.25)
ax_r2.set_axisbelow(True)

for lado in ("top", "right"):
    ax_mae.spines[lado].set_visible(False)
    ax_r2.spines[lado].set_visible(False)

fig.suptitle("Comparação dos candidatos sob LOGO + one-hot", fontsize=13, y=1.01)
fig.tight_layout()

DIR_GRAFICOS = Path("../assets/graficos") if Path("../assets").is_dir() else Path("assets/graficos")
DIR_GRAFICOS.mkdir(parents=True, exist_ok=True)
CAMINHO_FIG = DIR_GRAFICOS / "comparacao_modelos_logo.png"
fig.savefig(CAMINHO_FIG, dpi=150, bbox_inches="tight", facecolor="white")
plt.show()

print(f"Figura salva em: {CAMINHO_FIG}")

### A escolha por região

A Seção 4.3 da documentação já estabeleceu que existe **um modelo por região**,
porque as sete correspondem a regimes térmicos distintos. Uma vez aceito isso,
escolher um único algoritmo para todas as sete deixa de ser exigência técnica.

O critério, retomado da Seção 4.4.4 da documentação: onde MAE e R² concordam, a
escolha é direta; onde divergem, o R² pesa mais quando sua diferença é grande —
por ser a métrica que informa se o modelo distingue configurações dentro da
região — e o MAE decide quando a diferença de R² é pequena.

A célula abaixo aplica esse critério em código, em vez de por inspeção, para
que o resultado mude sozinho caso os modelos sejam retreinados.

In [ ]:
LIMIAR_R2 = 0.05   # diferenca de R2 a partir da qual o R2 decide sozinho

linhas_escolha = []
for i, area in enumerate(AREAS):
    rotulo = ROTULOS_AREA[area]
    rf = rf_por_area[rf_por_area["key_area"] == rotulo].iloc[0]
    gb = gb_por_area[gb_por_area["key_area"] == rotulo].iloc[0]

    d_mae = rf["MAE"] - gb["MAE"]     # positivo => GB melhor
    d_r2 = gb["R2"] - rf["R2"]        # positivo => GB melhor

    if abs(d_r2) >= LIMIAR_R2:
        escolhido = "Gradient Boosting" if d_r2 > 0 else "Random Forest"
        motivo = f"R2 decide (dif. {abs(d_r2):.3f})"
    else:
        escolhido = "Gradient Boosting" if d_mae > 0 else "Random Forest"
        motivo = f"MAE decide (dif. {abs(d_mae):.2f} C)"

    linhas_escolha.append({
        "Regiao": rotulo,
        "MAE RF": rf["MAE"], "MAE GB": gb["MAE"],
        "R2 RF": rf["R2"],   "R2 GB": gb["R2"],
        "Escolhido": escolhido, "Motivo": motivo,
    })

escolha = pd.DataFrame(linhas_escolha)

print("ESCOLHA POR REGIAO - Random Forest x Gradient Boosting")
print("=" * 108)
print(escolha.round(4).to_string(index=False))

print()
placar = escolha["Escolhido"].value_counts()
for modelo, n in placar.items():
    print(f"  {modelo:20s} {n} regiao(oes)")

In [ ]:
# --- Confronto com o mapa que efetivamente roda em producao
CAMINHO_EXPORT = (Path("../scripts/export_models.py")
                  if Path("../scripts").is_dir() else Path("scripts/export_models.py"))

texto = CAMINHO_EXPORT.read_text(encoding="utf-8")
trecho = texto.split("AREA_ALGORITHMS = {")[1].split("}")[0]

producao = {}
for linha in trecho.strip().splitlines():
    if ":" not in linha:
        continue
    chave, valor = linha.split(":", 1)
    producao[chave.strip().strip('",')] = valor.strip().strip(' ,"')

ROTULO_PARA_CHAVE = {"random_forest": "Random Forest",
                     "gradient_boosting": "Gradient Boosting"}

print("ESTE NOTEBOOK x MAPA DE PRODUCAO (scripts/export_models.py)")
print("=" * 88)
divergencias = 0
for _, linha in escolha.iterrows():
    regiao = linha["Regiao"]
    em_producao = ROTULO_PARA_CHAVE.get(producao.get(regiao, ""), "(ausente)")
    igual = em_producao == linha["Escolhido"]
    divergencias += (not igual)
    print(f"  {regiao:16s} notebook: {linha['Escolhido']:18s} "
          f"producao: {em_producao:18s} {'OK' if igual else '<-- DIVERGE'}")

print()
print(f"Divergencias: {divergencias}")

### Quanto o roteamento depende do limiar

O critério acima usa `LIMIAR_R2 = 0,05` para decidir quando o R² fala mais alto
que o MAE. Esse valor é uma escolha, não um resultado — então convém saber
quais regiões dependem dele e quais não.

In [ ]:
print("ROTEAMENTO SOB DIFERENTES LIMIARES DE R2")
print("=" * 96)

cabecalho = f"{'limiar':>8}  " + "  ".join(f"{ROTULOS_AREA[a][:13]:>13}" for a in AREAS)
print(cabecalho)

roteamentos = {}
for limiar in [0.00, 0.02, 0.05, 0.10, 0.20]:
    linha = []
    for area in AREAS:
        rotulo = ROTULOS_AREA[area]
        rf = rf_por_area[rf_por_area["key_area"] == rotulo].iloc[0]
        gb = gb_por_area[gb_por_area["key_area"] == rotulo].iloc[0]
        d_r2 = gb["R2"] - rf["R2"]
        d_mae = rf["MAE"] - gb["MAE"]
        if abs(d_r2) >= limiar:
            linha.append("GB" if d_r2 > 0 else "RF")
        else:
            linha.append("GB" if d_mae > 0 else "RF")
    roteamentos[limiar] = linha
    print(f"{limiar:8.2f}  " + "  ".join(f"{v:>13}" for v in linha))

print()
print("ESTABILIDADE POR REGIAO")
print("=" * 96)
for j, area in enumerate(AREAS):
    valores = {roteamentos[k][j] for k in roteamentos}
    marca = "estavel" if len(valores) == 1 else "SENSIVEL ao limiar"
    print(f"  {ROTULOS_AREA[area]:16s} {marca}")

**Leitura.** Quatro das sete regiões — Coil Temps, CT Knobs, IPC Comp e
IPC Cooling — dão o mesmo resultado sob qualquer limiar testado. A escolha ali
não depende do critério, e sim da distância real entre os dois algoritmos.

As outras três dependem, e em graus diferentes:

- **Oven HMI** é a mais frágil: vira com limiar 0,02. As duas métricas discordam
  e ambas as margens são mínimas (0,03 °C no MAE, 0,018 no R²). Dizer que "o
  Gradient Boosting vence em Oven HMI" seria dar a um empate a aparência de uma
  decisão.
- **Mainback e Main Vent Sys** viram com limiar 0,10. Esse limiar é alto — exige
  ignorar diferenças de R² de até 0,1 —, mas não é absurdo, então as duas não
  podem ser chamadas de estáveis sem ressalva. Nas duas, MAE e R² apontam para
  lados opostos, e o resultado registrado depende de qual métrica desempata.

**O que isso significa para o roteamento.** O limiar de 0,05 adotado acima não é
um resultado do experimento: é uma escolha de quem analisa, feita antes de ver o
efeito. Ela decide três das sete regiões. Onde o comparativo não separa os dois
algoritmos com folga, a decisão deve se apoiar em outros critérios — custo de
manter um artefato a mais, coerência com regiões vizinhas, estabilidade entre
retreinos — e ser registrada como tal, não como vantagem de desempenho medida.

### Por que o Decision Tree fica fora da decisão

O Decision Tree não entra no confronto acima por não superar o baseline de
forma consistente — o resultado impresso na Seção 5 mostra em quantas regiões
cada modelo o supera. Vale lembrar que a árvore teve aqui uma **vantagem
informacional** sobre os outros dois: seus hiperparâmetros foram buscados sobre
todas as configurações da região, inclusive as que depois serviriam de teste no
LOGO. Ficar atrás mesmo com essa vantagem reforça a conclusão.

Seu valor no projeto é outro: a leitura direta das regras, tratada a seguir.

## 6. Explicabilidade

A entrega exige que ao menos um dos modelos permita explicabilidade. Aqui há
duas formas complementares, e elas respondem a perguntas diferentes.

**A árvore de decisão** se lê diretamente: cada previsão é um caminho de regras
do tipo `variável ≤ limiar`. É a explicação mais transparente possível, porque
é o próprio modelo, não uma aproximação dele.

**A importância de variáveis** dos modelos de conjunto mostra quanto cada
variável de configuração pesou nas decisões — disponível nos três algoritmos
sem biblioteca adicional. Os notebooks de origem estendem essa leitura com
SHAP, que decompõe **previsões individuais** em contribuições por variável.

In [ ]:
# --- Importancia das variaveis, agregada das colunas one-hot de volta para a
# variavel de origem (cada variavel virou varias colunas binarias).
def importancia_por_variavel(modelo, colunas_onehot):
    """Soma a importancia das colunas one-hot de cada variavel de origem."""
    if hasattr(modelo, "feature_importances_"):
        bruta = modelo.feature_importances_
    else:  # MultiOutputRegressor: media entre os regressores por termopar
        bruta = np.mean([e.feature_importances_ for e in modelo.estimators_], axis=0)

    acumulado = {v: 0.0 for v in VARIAVEIS_INDEPENDENTES}
    for coluna, peso in zip(colunas_onehot, bruta):
        for variavel in VARIAVEIS_INDEPENDENTES:
            if coluna.startswith(variavel + "_"):
                acumulado[variavel] += float(peso)
                break
    return acumulado


linhas_imp = []
for area in AREAS:
    X, Y, _ = DADOS[area]
    modelo = RandomForestRegressor(random_state=SEMENTE, n_jobs=-1,
                                   **PARAMS_RF[area]).fit(X.values, Y.values)
    imp = importancia_por_variavel(modelo, X.columns)
    total = sum(imp.values()) or 1.0
    linhas_imp.append({"key_area": ROTULOS_AREA[area],
                       **{v: 100 * p / total for v, p in imp.items()}})

importancias = pd.DataFrame(linhas_imp).set_index("key_area")

print("PARTICIPACAO DE CADA VARIAVEL NA IMPORTANCIA TOTAL, POR REGIAO (%)")
print("=" * 104)
print(importancias.round(1).to_string())

In [ ]:
# --- Mapa de calor: qual variavel decide em cada regiao
fig, ax = plt.subplots(figsize=(11, 5))

matriz = importancias.values
imagem = ax.imshow(matriz, cmap="YlGnBu", aspect="auto")

ax.set_xticks(range(len(importancias.columns)))
ax.set_xticklabels(importancias.columns, rotation=35, ha="right", fontsize=9.5)
ax.set_yticks(range(len(importancias.index)))
ax.set_yticklabels(importancias.index, fontsize=9.5)

for i in range(matriz.shape[0]):
    for j in range(matriz.shape[1]):
        valor = matriz[i, j]
        ax.text(j, i, f"{valor:.0f}", ha="center", va="center", fontsize=9,
                color="white" if valor > matriz.max() * 0.55 else "#222222")

ax.set_title("Participação de cada variável na importância total, por região (%)\n"
             "Random Forest, importância por impureza agregada das colunas one-hot",
             fontsize=11.5, pad=12)
fig.colorbar(imagem, ax=ax, shrink=0.85, label="% da importância da região")
fig.tight_layout()

CAMINHO_IMP = DIR_GRAFICOS / "comparacao_importancia_por_area.png"
fig.savefig(CAMINHO_IMP, dpi=150, bbox_inches="tight", facecolor="white")
plt.show()

print(f"Figura salva em: {CAMINHO_IMP}")

# Qual variavel lidera em cada regiao
print()
print("VARIAVEL DOMINANTE POR REGIAO")
print("=" * 62)
for regiao in importancias.index:
    linha = importancias.loc[regiao]
    print(f"  {regiao:16s} {linha.idxmax():24s} {linha.max():5.1f}%")

In [ ]:
# --- A arvore de Mainback, lida como regras
from sklearn.tree import export_text

X_mb, Y_mb, alvos_mb = DADOS["mainback"]
arvore = DecisionTreeRegressor(random_state=SEMENTE,
                               **PARAMS_DT["mainback"]).fit(X_mb.values, Y_mb.values)

print(f"ARVORE DE DECISAO - Mainback (profundidade {arvore.get_depth()}, "
      f"{arvore.get_n_leaves()} folhas)")
print("=" * 78)
print(export_text(arvore, feature_names=list(X_mb.columns), max_depth=3))

### O que essa leitura **não** autoriza a dizer

A hipótese H2 (Seção 4.2.4) mostrou que as sete variáveis de configuração estão
**confundidas** no desenho dos ensaios: variar uma quase sempre implicou variar
outra. Diante de informação redundante, o modelo distribui o crédito entre as
variáveis de forma arbitrária.

Por isso, uma variável com importância baixa significa que o modelo **não
precisou** dela para prever — não que ela seja irrelevante para o aquecimento
do fogão. A leitura descreve **como o modelo decide**, não **como o fogão
aquece**.

## 7. Conclusão

In [ ]:
print("RESUMO - LOGO, one-hot, mediana entre sensores, media das sete regioes")
print("=" * 88)
print(resumo_geral.round(3).to_string(index=False))

print()
print("ROTEAMENTO POR REGIAO")
print("=" * 88)
print(escolha[["Regiao", "Escolhido", "Motivo"]].to_string(index=False))

print()
print(f"Placar: " + ", ".join(f"{m} em {n}" for m, n in placar.items()))

### O que este notebook estabelece

1. **Os três candidatos foram comparados sob um protocolo único** — mesma base,
   mesma codificação, mesma validação e, pela primeira vez, a mesma agregação
   de métricas. A inconsistência média/mediana entre os notebooks de origem
   está medida na Seção 3 e neutralizada daqui em diante.

2. **Random Forest e Gradient Boosting são os candidatos viáveis.** O Decision
   Tree fica atrás mesmo tendo recebido uma vantagem no tuning, e seu papel no
   projeto é a leitura direta das regras.

3. **O roteamento por região é reproduzível.** A Seção 5 aplica o critério em
   código e confronta o resultado com o mapa `AREA_ALGORITHMS` que roda em
   produção, de modo que documentação e sistema não podem divergir em silêncio.

### O que ele não resolve

**A base continua pequena.** São 32 configurações por região — não 320. Sob
LOGO a medição é mais estável do que sob o corte fixo, mas margens de décimos
de grau entre dois algoritmos não sustentam, sozinhas, uma decisão de
roteamento. As regiões em que a diferença é mínima estão marcadas na coluna
"Motivo" da tabela de escolha.

**O poder de distinção é modesto.** Mesmo os melhores R² indicam que os modelos
acertam bem o **patamar térmico** de cada região, e ainda não escolhem com
segurança entre duas configurações parecidas. Na prática, isso os qualifica
para priorizar quais ensaios físicos rodar primeiro — não para substituí-los.

**A recomendação ao parceiro segue a mesma:** que os próximos ensaios sejam
planejados para cobrir combinações ainda não testadas, em vez de repetir as já
frequentes. Nenhum ajuste de hiperparâmetro substitui dados novos.

### Sobre reprodutibilidade

Os valores impressos aqui podem diferir na terceira casa decimal dos
registrados nos notebooks de origem, que rodaram em outra versão do
interpretador. A comparação **interna** deste notebook não é afetada: os
quatro modelos são avaliados no mesmo processo, sobre os mesmos dados, com a
mesma semente. É essa comparação interna, e não o valor absoluto de cada
célula, que sustenta a escolha por região.